In [44]:
import os
os.chdir(r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code')

In [45]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [46]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [47]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [48]:
model1 = MLP()
model1.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_4 (Flatten)         (None, 504)               0         
                                                                 
 dense_8 (Dense)             (None, 32)                16160     
                                                                 
 dense_9 (Dense)             (None, 1)                 33        
                                                                 
Total params: 16193 (63.25 KB)
Trainable params: 16193 (63.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [49]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [50]:
checkpoints = r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [51]:
os.path.exists(JSON_PATH)

True

In [52]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [53]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [54]:
import os
path_dataset =r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [55]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.012532234191894531 sec


In [56]:
train_X.shape

(835, 24, 21)

In [57]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/10
18/27 [===================>..........] - ETA: 0s - loss: 0.3229 - mae: 0.3229 - mape: 144.3098 
Epoch 1: val_loss improved from inf to 0.08903, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.09.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 2s 26ms/step - loss: 0.2590 - mae: 0.2590 - mape: 123.2374 - val_loss: 0.0890 - val_mae: 0.0890 - val_mape: 30.9150
Epoch 2/10
18/27 [===================>..........] - ETA: 0s - loss: 0.0789 - mae: 0.0789 - mape: 39.4137
Epoch 2: val_loss improved from 0.08903 to 0.06283, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0002-loss0.06.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 16ms/step - loss: 0.0751 - mae: 0.0751 - mape: 40.1415 - val_loss: 0.0628 - val_mae: 0.0628 - val_mape: 21.6802
Epoch 3/10
17/27 [=================>............] - ETA: 0s - loss: 0.0711 - mae: 0.0711 - mape: 33.7643
Epoch 3: val_loss did not improve from 0.06283
27/27 [==============================] - 0s 17ms/step - loss: 0.0676 - mae: 0.0676 - mape: 36.8133 - val_loss: 0.0682 - val_mae: 0.0682 - val_mape: 20.9325
Epoch 4/10
18/27 [===================>..........] - ETA: 0s - loss: 0.0608 - mae: 0.0608 - mape: 30.0307
Epoch 4: val_loss did not improve from 0.06283
27/27 [==============================] - 0s 15ms/step - loss: 0.0577 - mae: 0.0577 - mape: 30.4205 - val_loss: 0.1002 - val_mae: 0.1002 - val_mape: 35.5880
Epoch 5/10
17/27 [=================>............] - ETA: 0s - loss: 0.0499 - mae: 0.0499 - mape: 25.9940
Epoch 5: val_loss did not improve from 0.06283
27/27 [==============================] - 0s 14ms/step - loss: 0.0556 - mae: 

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 17ms/step - loss: 0.0584 - mae: 0.0584 - mape: 30.1577 - val_loss: 0.0494 - val_mae: 0.0494 - val_mape: 18.7132
Epoch 8/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0450 - mae: 0.0450 - mape: 22.5291
Epoch 8: val_loss did not improve from 0.04940
27/27 [==============================] - 0s 17ms/step - loss: 0.0454 - mae: 0.0454 - mape: 22.1057 - val_loss: 0.0675 - val_mae: 0.0675 - val_mape: 24.5927
Epoch 9/10
19/27 [====================>.........] - ETA: 0s - loss: 0.0516 - mae: 0.0516 - mape: 24.9707
Epoch 9: val_loss improved from 0.04940 to 0.04668, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0009-loss0.05.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 16ms/step - loss: 0.0535 - mae: 0.0535 - mape: 27.2490 - val_loss: 0.0467 - val_mae: 0.0467 - val_mape: 13.9738
Epoch 10/10
23/27 [========================>.....] - ETA: 0s - loss: 0.0495 - mae: 0.0495 - mape: 24.4677
Epoch 10: val_loss did not improve from 0.04668
27/27 [==============================] - 0s 18ms/step - loss: 0.0489 - mae: 0.0489 - mape: 24.3776 - val_loss: 0.1057 - val_mae: 0.1057 - val_mape: 35.9053


In [58]:

model = load_model(r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.05.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 102ms/step
Mean Absolute Error (MAE): 1223.96
Median Absolute Error (MedAE): 1299.48
Mean Squared Error (MSE): 1800927.67
Root Mean Squared Error (RMSE): 1341.99
Mean Absolute Percentage Error (MAPE): 7.79 %
Median Absolute Percentage Error (MDAPE): 8.42 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# Fine Tuning

In [59]:
checkpoints = r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.05.h5'
model=r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.05.h5'
start_epoch= 56

In [60]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.05.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [61]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
13/27 [=============>................] - ETA: 0s - loss: 0.0556 - mae: 0.0556 - mape: 29.3854  
Epoch 1: val_loss improved from inf to 0.04911, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.05.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 27ms/step - loss: 0.0545 - mae: 0.0545 - mape: 30.5754 - val_loss: 0.0491 - val_mae: 0.0491 - val_mape: 15.8343
Epoch 2/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0527 - mae: 0.0527 - mape: 29.2524
Epoch 2: val_loss improved from 0.04911 to 0.04908, saving model to C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.05.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 18ms/step - loss: 0.0525 - mae: 0.0525 - mape: 28.8829 - val_loss: 0.0491 - val_mae: 0.0491 - val_mape: 15.8030
Epoch 3/10
12/27 [============>.................] - ETA: 0s - loss: 0.0541 - mae: 0.0541 - mape: 22.1736
Epoch 3: val_loss did not improve from 0.04908
27/27 [==============================] - 0s 16ms/step - loss: 0.0517 - mae: 0.0517 - mape: 28.9759 - val_loss: 0.0497 - val_mae: 0.0497 - val_mape: 16.1040
Epoch 4/10
15/27 [===============>..............] - ETA: 0s - loss: 0.0506 - mae: 0.0506 - mape: 27.1072
Epoch 4: val_loss did not improve from 0.04908
27/27 [==============================] - 0s 16ms/step - loss: 0.0506 - mae: 0.0506 - mape: 27.9180 - val_loss: 0.0494 - val_mae: 0.0494 - val_mape: 16.0183
Epoch 5/10
13/27 [=============>................] - ETA: 0s - loss: 0.0489 - mae: 0.0489 - mape: 24.9235
Epoch 5: val_loss did not improve from 0.04908
27/27 [==============================] - 0s 17ms/step - loss: 0.0500 - mae: 

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 18ms/step - loss: 0.0488 - mae: 0.0488 - mape: 26.5509 - val_loss: 0.0474 - val_mae: 0.0474 - val_mape: 15.3750
Epoch 9/10
15/27 [===============>..............] - ETA: 0s - loss: 0.0484 - mae: 0.0484 - mape: 17.1331
Epoch 9: val_loss did not improve from 0.04742
27/27 [==============================] - 0s 16ms/step - loss: 0.0477 - mae: 0.0477 - mape: 25.5414 - val_loss: 0.0537 - val_mae: 0.0537 - val_mape: 17.6521
Epoch 10/10
13/27 [=============>................] - ETA: 0s - loss: 0.0474 - mae: 0.0474 - mape: 23.7769
Epoch 10: val_loss did not improve from 0.04742
27/27 [==============================] - 0s 17ms/step - loss: 0.0470 - mae: 0.0470 - mape: 24.5692 - val_loss: 0.0533 - val_mae: 0.0533 - val_mape: 17.7236


In [62]:

model = load_model(r'C:\wajeeh\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0004-loss0.05.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 119ms/step
Mean Absolute Error (MAE): 576.83
Median Absolute Error (MedAE): 455.19
Mean Squared Error (MSE): 482882.26
Root Mean Squared Error (RMSE): 694.9
Mean Absolute Percentage Error (MAPE): 3.66 %
Median Absolute Percentage Error (MDAPE): 2.94 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
